# ADIRR — WCNPS 2026
## Notebook experimental v0.4

Infraestrutura executavel do primeiro experimento parcial da ADIRR.

**Escopo:** D0 -> D1 -> D2 -> D3 -> D4 -> D6.

Novidades da v0.4 em relacao a v0.3:
- D6 implementado (Eq. 7): mapeia B em rho publicavel, com V (validade), v_I, theta_I, pi_t e proveniencia propagados de D4. Ver `OPERACIONALIZACAO_D6_v0_1.md`.
- Correcao estrutural: `method` (M1/M2) e `L` agora sao colunas explicitas nos DataFrames de D3/D4/D6, nao apenas chaves de dicionario, permitindo concatenar a grade completa (L x q x metodo) em uma unica tabela.
- Regra de `uncertainty` (D4) confirmada em 2026-09-28: binaria, `1` apenas se `evidence_mass=0`. Ver `OPERACIONALIZACAO_D4_v0_1.md`.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Tuple, Any
import math, sys, platform
import numpy as np
import pandas as pd
print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| Pandas", pd.__version__)

## 1. Configuracao do protocolo

L passa a ser um eixo experimental (L_levels), nao mais um unico valor fixo.

In [ ]:
@dataclass(frozen=True)
class ProtocolConfig:
    master_seed: int = 20260922
    T: int = 1000
    L_levels: Tuple[int, ...] = (25, 50, 100)
    q_levels: Tuple[float, ...] = (1.00, 0.80, 0.60, 0.40)
    pilot_replications: int = 100
    scientific_replications: int = 2000
    mcse_ratio_target: float = 0.025
    neutral_weight_positive: float = 0.5
    neutral_weight_negative: float = 0.5

CFG = ProtocolConfig()
for _L in CFG.L_levels:
    assert CFG.T % _L == 0
CFG

## 2. Streams e mascaras aninhadas

In [ ]:
def child_rng(master_seed: int, replication_id: int) -> np.random.Generator:
    ss = np.random.SeedSequence(master_seed, spawn_key=(replication_id,))
    return np.random.default_rng(ss)

def generate_uniform_stream(replication_id: int, cfg: ProtocolConfig = CFG) -> np.ndarray:
    return child_rng(cfg.master_seed, replication_id).random(cfg.T)

def nested_observation_masks(u: np.ndarray, q_levels=CFG.q_levels):
    masks = {q: (u <= q) for q in q_levels}
    ordered = sorted(q_levels)
    for q_low, q_high in zip(ordered[:-1], ordered[1:]):
        assert np.all(masks[q_low] <= masks[q_high])
    return masks

## 3. D0 e D1 — comportamento cooperativo determinístico e validação

In [ ]:
def generate_d0_events(replication_id: int, cfg: ProtocolConfig = CFG) -> pd.DataFrame:
    t = np.arange(1, cfg.T + 1)
    return pd.DataFrame({
        "replication_id": replication_id,
        "t": t,
        "opportunity_id": [f"r{replication_id:04d}_t{x:04d}" for x in t],
        "agent_id": "agent_coop_01",
        "context": "ctx_01",
        "behavior": np.ones(cfg.T, dtype=np.int8),
        "source": "synthetic_ground_truth",
    })

def validate_d1(events: pd.DataFrame, cfg: ProtocolConfig = CFG) -> pd.DataFrame:
    assert len(events) == cfg.T
    assert events["opportunity_id"].is_unique
    assert (events["behavior"] == 1).all()
    out = events.copy()
    out["d1_valid"] = True
    return out

## 4. D2 — observabilidade e janelas sobre oportunidades

`assign_d2_windows` agora recebe `L` como parametro explicito, em vez de usar um unico valor fixo em `CFG`. Isso e o que habilita o desenho fatorial q x L sem alterar D0, D1, D3 ou D4.

In [ ]:
def attach_observability(events, mask, q):
    out = events.copy()
    out["q_nominal"] = q
    out["observed"] = np.asarray(mask, dtype=bool)
    return out

def assign_d2_windows(events, L: int, cfg: ProtocolConfig = CFG):
    out = events.copy()
    out["window_length"] = L
    out["window_id"] = ((out["t"] - 1) // L).astype(int)
    out["window_start"] = out["window_id"] * L + 1
    out["window_end"] = out["window_start"] + L - 1
    assert out["window_id"].nunique() == cfg.T // L
    return out

## 5. M1 x M2 — adaptador experimental D2->D3

M1 preserva a ausencia. M2 insere eta0=(0,5;0,5) apenas para oportunidades nao observadas. A pseudo-evidencia nao e registrada como evento real em D0/D1. Inalterado em relacao a v0.2.

In [ ]:
def prepare_m1_input(d2_events):
    out = d2_events.copy()
    out["evidence_kind"] = np.where(out["observed"], "real", "missing")
    out["support_positive"] = np.where(out["observed"], out["behavior"].astype(float), np.nan)
    out["support_negative"] = np.where(out["observed"], 1.0-out["behavior"].astype(float), np.nan)
    out["experimental_source"] = np.where(out["observed"], "real_observation", "explicit_absence")
    return out

def prepare_m2_input(d2_events, cfg: ProtocolConfig = CFG):
    out = d2_events.copy()
    out["evidence_kind"] = np.where(out["observed"], "real", "neutral_imputation")
    out["support_positive"] = np.where(out["observed"], out["behavior"].astype(float), cfg.neutral_weight_positive)
    out["support_negative"] = np.where(out["observed"], 1.0-out["behavior"].astype(float), cfg.neutral_weight_negative)
    out["experimental_source"] = np.where(out["observed"], "real_observation", "neutral_imputation_baseline")
    return out

## 6. Invariantes pré-D3

In [ ]:
def check_pre_d3_invariants(m1, m2, q):
    checks = {
        "same_opportunities": m1["opportunity_id"].equals(m2["opportunity_id"]),
        "same_mask": m1["observed"].equals(m2["observed"]),
        "neutral_count_ok": (m2["evidence_kind"]=="neutral_imputation").sum() == (~m2["observed"]).sum(),
        "no_neutral_in_m1": (m1["evidence_kind"]=="neutral_imputation").sum() == 0,
        "o100_no_missing": True if q != 1.0 else bool(m1["observed"].all() and m2["observed"].all()),
    }
    assert all(checks.values()), checks
    return checks

## 7. D3 — instanciacao de phi (Eq. 4)

Inalterado em relacao a v0.2. Ver `OPERACIONALIZACAO_D3_v0_1.md`.

In [ ]:
D3_FEATURE_VERSION = "phi_binary_features_v0.1"

def run_d3(d2_to_d3: pd.DataFrame, *, theta_phi: Dict[str, Any] | None = None) -> pd.DataFrame:
    """Instanciacao experimental de phi para o cenario binario estacionario."""
    theta_phi = dict(theta_phi or {})
    required = {
        "replication_id", "q_nominal", "window_id", "window_start", "window_end",
        "observed", "support_positive", "support_negative", "evidence_kind"
    }
    missing = required.difference(d2_to_d3.columns)
    assert not missing, f"Entrada D3 incompleta: {missing}"

    df = d2_to_d3.copy()
    df["positive_support_filled"] = df["support_positive"].fillna(0.0)
    df["negative_support_filled"] = df["support_negative"].fillna(0.0)
    df["neutral_imputation"] = (df["evidence_kind"] == "neutral_imputation").astype(int)

    group_cols = [
        "replication_id", "q_nominal", "window_id", "window_start", "window_end"
    ]
    z = (df.groupby(group_cols, as_index=False)
           .agg(
               n_opportunities=("observed", "size"),
               n_observed=("observed", "sum"),
               positive_support=("positive_support_filled", "sum"),
               negative_support=("negative_support_filled", "sum"),
               n_neutral_imputations=("neutral_imputation", "sum"),
           ))

    z["n_missing"] = z["n_opportunities"] - z["n_observed"]
    z["z_observability_rate"] = z["n_observed"] / z["n_opportunities"]
    z["evidence_mass"] = z["positive_support"] + z["negative_support"]

    z["z_success_rate"] = np.where(
        z["evidence_mass"] > 0,
        z["positive_support"] / z["evidence_mass"],
        np.nan,
    )
    z["z_failure_rate"] = np.where(
        z["evidence_mass"] > 0,
        z["negative_support"] / z["evidence_mass"],
        np.nan,
    )

    z["theta_phi"] = [theta_phi.copy() for _ in range(len(z))]
    z["d3_feature_version"] = D3_FEATURE_VERSION

    assert np.all((z["z_observability_rate"] >= 0) & (z["z_observability_rate"] <= 1))
    valid_mass = z["evidence_mass"] > 0
    assert np.allclose(
        (z.loc[valid_mass, "z_success_rate"] + z.loc[valid_mass, "z_failure_rate"]).values, 1.0
    )
    return z

## 8. D4 — instanciacao de I_theta_I (Eq. 5)

**Novo na v0.3.** Ver `OPERACIONALIZACAO_D4_v0_1.md` para a justificativa completa.

D4 produz `B_{a,c,t}` como tupla explicita e nao colapsada:
`(support, counter_support, uncertainty, observability)`, mais `G_prov` e `pi_t` explicitos.

Regras respeitadas:
- nao transforma `z` em reputacao por conveniencia (D4 nao produz `rho`);
- `E_dir = E_ind = vazio` nesta etapa, declarados explicitamente, nao omitidos;
- `G_prov` representado como fonte unica, sem dependencias;
- `pi_t` e um objeto de configuracao explicito, nao implicito no codigo;
- nenhum modelo externo nomeado (Beta, Subjective Logic, media) e importado como definicao do operador.

In [ ]:
D4_VERSION = "I_thetaI_binary_v0.1"

DEFAULT_POLICY = {
    "window_semantics": "tumbling",
    "direct_evidence_policy": "excluded_encoded_in_z",
    "indirect_evidence_policy": "excluded_strategic_decision",
    "conflict_rule": "support_and_counter_support_denote_conflict_not_resolved_here",
}

def run_d4(z_df: pd.DataFrame, *, theta_I: Dict[str, Any] | None = None) -> pd.DataFrame:
    """Instanciacao experimental de I_thetaI para o cenario binario, fonte unica,
    sem evidencia direta/indireta adicional. Nao introduz modelo externo."""
    theta_I = dict(theta_I or {})
    required = {"positive_support", "negative_support", "evidence_mass", "z_observability_rate"}
    missing = required.difference(z_df.columns)
    assert not missing, f"Entrada D4 incompleta: {missing}"

    df = z_df.copy()

    # E_dir, E_ind explicitamente vazios nesta etapa (ver OPERACIONALIZACAO_D4_v0_1.md)
    df["support"] = df["positive_support"]
    df["counter_support"] = df["negative_support"]
    df["observability"] = df["z_observability_rate"]
    df["uncertainty"] = np.where(df["evidence_mass"] == 0, 1.0, 0.0)

    n = len(df)
    df["G_prov"] = [{"sources": ["agent_coop_01_observer"], "dependency_edges": []} for _ in range(n)]
    policy = dict(DEFAULT_POLICY)
    policy.update(theta_I.get("policy", {}))
    df["pi_t"] = [dict(policy) for _ in range(n)]
    df["d4_version"] = D4_VERSION

    assert set(df["uncertainty"].unique()).issubset({0.0, 1.0})
    assert np.all((df["observability"] >= 0) & (df["observability"] <= 1))
    return df

## 9. D6 — instanciação da publicação reputacional (Eq. 7)

**Novo na v0.4.** Ver `OPERACIONALIZACAO_D6_v0_1.md` para a justificativa completa.

D6 mapeia `B` (produzido por D4) no registro publicável `R^(v) = (rho, B, V, v_I, theta_I, pi_t, p)`.

Única quantidade numérica nova: `rho = support / (support + counter_support)` quando `evidence_mass > 0`; caso contrário `rho = NaN` e a janela é marcada `valid=False`. Nenhum modelo externo nomeado foi usado para definir `rho`; `uncertainty` e `observability` permanecem como campos próprios de `B`, não absorvidos pela fórmula de `rho`.

In [ ]:
D6_VERSION = "R_v_binary_v0.1"
REQUIRED_D6 = {"replication_id","method","q_nominal","window_id","rho"}

def run_d6(b_df: pd.DataFrame, *, theta_I: Dict[str, Any] | None = None) -> pd.DataFrame:
    """Instanciacao experimental de D6 (Eq. 7): mapeia B em rho publicavel.
    Nao redefine support/counter_support/uncertainty/observability, apenas
    os propaga junto com o novo escalar rho e os metadados de validade."""
    theta_I = dict(theta_I or {})
    required = {"support", "counter_support", "uncertainty", "observability", "G_prov", "pi_t", "d4_version"}
    missing = required.difference(b_df.columns)
    assert not missing, f"Entrada D6 incompleta: {missing}"

    df = b_df.copy()
    evidence_mass = df["support"] + df["counter_support"]
    valid = evidence_mass > 0

    df["rho"] = np.where(valid, df["support"] / evidence_mass.where(valid, np.nan), np.nan)
    df["valid"] = valid
    df["invalid_reason"] = np.where(valid, None, "empty_window")
    df["v_I"] = df["d4_version"] + "|" + D6_VERSION
    df["theta_I"] = [theta_I.copy() for _ in range(len(df))]
    df["p_provenance"] = df["G_prov"]
    df["d6_version"] = D6_VERSION

    assert np.all(df.loc[valid, "rho"].between(0, 1))
    assert df.loc[~valid, "rho"].isna().all()
    return df

def check_d6_contract(d6_df: pd.DataFrame):
    missing = REQUIRED_D6.difference(d6_df.columns)
    assert not missing, f"Contrato D6 nao satisfeito, faltando: {missing}"
    return True

def add_o100_reference(d6):
    check_d6_contract(d6)
    ref = (d6[d6["q_nominal"]==1.0][["replication_id","method","L","window_id","rho"]]
           .rename(columns={"rho":"rho_o100"}))
    out = d6.merge(ref,on=["replication_id","method","L","window_id"],how="left",validate="many_to_one")
    out["D"] = out["rho"] - out["rho_o100"]
    return out

def mcse(values):
    s = pd.Series(values).dropna()
    return s.std(ddof=1)/math.sqrt(len(s))

def summarize_replication_metrics(df):
    return (df.groupby(["replication_id","method","q_nominal","L"],as_index=False)
              .agg(DMA_r=("D","mean"), MAE_r=("D",lambda s: np.abs(s).mean())))

## 10. Construção de uma replicação através da grade q x L, até D4

Substitui `build_replication_pre_d3` da v0.2. Agora percorre `L_levels x q_levels` e executa D0->D1->D2->M1/M2->D3->D4 para cada combinação.

In [ ]:
def build_replication_through_d4(replication_id: int, cfg: ProtocolConfig = CFG):
    d0 = generate_d0_events(replication_id, cfg)
    d1 = validate_d1(d0, cfg)
    u = generate_uniform_stream(replication_id, cfg)
    masks = nested_observation_masks(u, cfg.q_levels)

    results = {}
    for L in cfg.L_levels:
        for q in cfg.q_levels:
            d2 = assign_d2_windows(attach_observability(d1, masks[q], q), L, cfg)
            m1_in, m2_in = prepare_m1_input(d2), prepare_m2_input(d2, cfg)
            check_pre_d3_invariants(m1_in, m2_in, q)
            z_m1, z_m2 = run_d3(m1_in), run_d3(m2_in)
            b_m1, b_m2 = run_d4(z_m1), run_d4(z_m2)
            # Rotulagem explicita de method e L como colunas (nao alteram nenhum calculo de D3/D4)
            b_m1 = b_m1.assign(method="M1", L=L)
            b_m2 = b_m2.assign(method="M2", L=L)
            results[(L, q)] = {"M1": b_m1, "M2": b_m2}
    return results

rep0 = build_replication_through_d4(0)
len(rep0)

## 10.1 Executar D6 sobre a grade completa (uma replicação)

Aplica `run_d6` a cada condição (L, q, método) da replicação já construída até D4, produzindo a tabela flat necessária para `add_o100_reference` e as métricas.

In [ ]:
def run_d6_over_grid(results_d4: dict, *, theta_I: Dict[str, Any] | None = None) -> pd.DataFrame:
    frames = []
    for (L, q), methods in results_d4.items():
        for m, b_df in methods.items():
            d6_df = run_d6(b_df, theta_I=theta_I)
            frames.append(d6_df)
    out = pd.concat(frames, ignore_index=True)
    check_d6_contract(out)
    return out

d6_rep0 = run_d6_over_grid(rep0)
d6_rep0[["replication_id","method","L","q_nominal","window_id","rho","valid","uncertainty","observability"]].head(10)

## 11. Smoke tests de D3 + D4 + D6

Verificam, para toda combinação de L e q:
1. em O100, M1 = M2 em support/counter_support/observability/uncertainty **e em rho** (invariante obrigatório do protocolo, agora estendido ao registro publicável);
2. em M2, `uncertainty` é sempre 0 e `rho` é sempre válido (nunca NaN), pois toda ausência recebe massa neutra;
3. em M1/O40, `uncertainty=1` (e `rho` NaN) só ocorre em janelas com `evidence_mass=0` — consistente e estrutural, não forçado;
4. `rho` permanece em `[0,1]` em toda janela válida.

In [ ]:
def test_o100_m1_equals_m2(d6_df, cfg: ProtocolConfig = CFG):
    o100 = d6_df[d6_df["q_nominal"] == 1.00]
    for L in cfg.L_levels:
        sub = o100[o100["L"] == L]
        m1 = sub[sub["method"]=="M1"].sort_values("window_id").reset_index(drop=True)
        m2 = sub[sub["method"]=="M2"].sort_values("window_id").reset_index(drop=True)
        for col in ["support", "counter_support", "observability", "uncertainty", "rho"]:
            assert np.allclose(m1[col].values, m2[col].values, equal_nan=True), (L, col)
    return True

def test_m2_uncertainty_always_zero_and_rho_valid(d6_df):
    m2 = d6_df[d6_df["method"]=="M2"]
    assert (m2["uncertainty"] == 0.0).all()
    assert m2["rho"].notna().all()
    assert m2["valid"].all()
    return True

def test_m1_uncertainty_and_rho_structural_at_o40(d6_df):
    sub = d6_df[(d6_df["method"]=="M1") & (d6_df["q_nominal"]==0.40)]
    empty_mask = np.isclose(sub["support"] + sub["counter_support"], 0.0)
    assert np.all(sub.loc[empty_mask, "uncertainty"] == 1.0)
    assert sub.loc[empty_mask, "rho"].isna().all()
    assert np.all(sub.loc[~empty_mask, "uncertainty"] == 0.0)
    assert sub.loc[~empty_mask, "rho"].notna().all()
    return True

def test_rho_bounds(d6_df):
    valid = d6_df[d6_df["valid"]]
    assert valid["rho"].between(0, 1).all()
    return True

{
    "o100_m1_eq_m2": test_o100_m1_equals_m2(d6_rep0),
    "m2_uncertainty_zero_rho_valid": test_m2_uncertainty_always_zero_and_rho_valid(d6_rep0),
    "m1_uncertainty_rho_structural_o40": test_m1_uncertainty_and_rho_structural_at_o40(d6_rep0),
    "rho_bounds": test_rho_bounds(d6_rep0),
}

## 12. Resumo do smoke test (grade completa L x q x método)

In [ ]:
def summarize_smoke(d6_df):
    return (d6_df.groupby(["L","q_nominal","method"], as_index=False)
              .agg(mean_support=("support","mean"),
                   mean_counter_support=("counter_support","mean"),
                   mean_observability=("observability","mean"),
                   mean_uncertainty=("uncertainty","mean"),
                   mean_rho=("rho","mean"),
                   n_invalid=("valid", lambda s: (~s).sum()),
                   n_windows=("rho","size"))
              .sort_values(["L","q_nominal","method"])
              .reset_index(drop=True))

smoke_summary = summarize_smoke(d6_rep0)
smoke_summary

## 13. Checks globais do protocolo

In [ ]:
def global_protocol_checks(cfg: ProtocolConfig = CFG):
    assert cfg.T == 1000
    assert set(cfg.L_levels) == {25, 50, 100}
    assert cfg.pilot_replications == 100
    assert cfg.scientific_replications == 2000
    assert 1/math.sqrt(cfg.scientific_replications) <= cfg.mcse_ratio_target
    out = {"scientific_replications": cfg.scientific_replications,
           "mcse_ratio_nominal": 1/math.sqrt(cfg.scientific_replications)}
    for L in cfg.L_levels:
        K = cfg.T // L
        p_empty_o40 = (1-0.40)**L
        out[f"L={L}"] = {
            "windows_per_replication": K,
            "p_empty_window_O40": p_empty_o40,
            "expected_empty_windows_O40_full_grid": cfg.scientific_replications*K*p_empty_o40,
        }
    return out

global_protocol_checks()

## 14. Próximos passos

1. ~~Operacionalizar D3.~~ Concluído (v0.2).
2. ~~Operacionalizar D4.~~ Concluído (v0.3); regra de `uncertainty` confirmada em 2026-09-28.
3. ~~Implementar D6 mínimo.~~ Concluído nesta versão (v0.4). `rho` propagado, `V`/validade e proveniência mapeados de `B`.
4. ~~Testar O100: M1=M2 sobre o registro D6 completo.~~ Validado nesta versão (inclui `rho`).
5. Rodar piloto técnico em escala (100 réplicas) sobre a grade completa L x q, como gate de decisão de prazo (ver `DESENHO_FATORIAL_Q_L_v0_1.md`).
6. Congelar código e parâmetros.
7. Executar réplicas científicas (R=2000, ou ajustado conforme piloto) sobre a grade L x q aprovada.

**O notebook agora produz `rho` e passa em todos os invariantes estruturais. Os números de uma única replicação (rep0) não são resultado científico — apenas validação de engenharia. O resultado científico depende do piloto técnico e da execução em R réplicas.**